[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/057.%20The%20Periodic%20Review%20%28Base-Stock%29%20Policy%20Problem/P57-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 57. The Periodic Review (Base-Stock) Policy Problem

## Tier 1: The Pen & Paper Method (Stochastic Programming Formulation)

### Key assumptions
- Demand follows a normal distribution with known mean and standard deviation
- Lead time is constant and known
- Review periods are fixed and regular
- Ordering policy is base-stock (order up to level S)
- All cost parameters are constant over time

### Approach (step-by-step)
1. **Mathematical Model**: Formulate the periodic review problem as a stochastic programming model
2. **Base-Stock Level Calculation**: Compute optimal base-stock level using service level requirements
3. **Cost Analysis**: Calculate expected ordering, holding, and stockout costs
4. **Optimization**: Find the review period and base-stock level that minimize total expected cost

### What to look for in the results
- Optimal base-stock level that balances holding and stockout costs
- Review period that minimizes total expected cost
- Service level achievement verification
- Cost breakdown showing the trade-offs between different cost components

### Concrete example (from the source)
MedSupply Distribution manages syringe inventory with:
- Weekly demand: μ = 12,000 units, σ = 2,400 units
- Lead time: L = 0.5 weeks
- Target service level: α = 0.95 (z₀.₉₅ = 1.645)
- Costs: h = $0.15/unit/week, K = $75/order, p = $8/unit

### Why this Tier exists vs earlier Tiers
This is the foundational tier that establishes the mathematical framework for periodic review inventory systems. It provides:
- **Theoretical foundation**: Mathematical formulation of the problem
- **Optimal solution**: Analytical methods for finding optimal parameters
- **Benchmark**: Reference point for comparing heuristic and advanced methods
- **Educational value**: Clear understanding of the underlying inventory theory

### Pros / Cons vs other approaches
**Pros:**
- Provides provably optimal solutions for the given assumptions
- Clear mathematical interpretation of results
- Fast computation for analytical solutions
- Establishes theoretical bounds on performance

**Cons:**
- Requires strong assumptions (normal demand, constant parameters)
- Limited to simple problem structures
- Cannot handle complex constraints or multiple objectives
- May not scale well to large, complex systems

### When to use this Tier
- When demand patterns are stable and predictable
- When you need theoretical optimal solutions
- For benchmarking other methods
- When computational efficiency is critical
- For educational purposes and understanding inventory theory

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from math import sqrt, inf
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
class BaseStockOptimizer:
    """
    Periodic Review Base-Stock Policy Optimizer
    
    This class implements the mathematical formulation for optimizing
    periodic review inventory policies with base-stock level control.
    """
    
    def __init__(self, demand_mean, demand_std, lead_time, holding_cost, 
                 ordering_cost, stockout_cost, service_level):
        """
        Initialize the optimizer with problem parameters.
        
        Parameters:
        - demand_mean: Mean demand per period (μ)
        - demand_std: Standard deviation of demand per period (σ)
        - lead_time: Supplier lead time in periods (L)
        - holding_cost: Holding cost per unit per period (h)
        - ordering_cost: Fixed ordering cost per order (K)
        - stockout_cost: Stockout penalty cost per unit (p)
        - service_level: Target service level (α)
        """
        self.mu = demand_mean
        self.sigma = demand_std
        self.L = lead_time
        self.h = holding_cost
        self.K = ordering_cost
        self.p = stockout_cost
        self.alpha = service_level
        # Calculate critical z-value for service level
        self.z_alpha = norm.ppf(service_level)
        
        print(f"Initialized Base Stock Optimizer:")
        print(f"  Demand: N({self.mu:,}, {self.sigma:,}²)")
        print(f"  Lead time: {self.L} weeks")
        print(f"  Service level: {self.alpha:.3f} (z = {self.z_alpha:.3f})")
        print(f"  Costs: h=${self.h}, K=${self.K}, p=${self.p}")
    
    def calculate_base_stock_level(self, review_period):
        """
        Calculate the optimal base-stock level for a given review period.
        
        Formula: S* = μ(R+L) + zα × σ × √(R+L)
        
        This ensures the base-stock level covers expected demand during
        the protection interval plus safety stock for the target service level.
        """
        protection_interval = review_period + self.L
        expected_demand = self.mu * protection_interval
        demand_std = self.sigma * sqrt(protection_interval)
        base_stock = expected_demand + self.z_alpha * demand_std
        
        return base_stock, protection_interval, expected_demand, demand_std
    
    def calculate_total_cost(self, review_period, base_stock_level):
        """
        Calculate expected total cost for given review period and base-stock level.
        
        Components:
        1. Ordering cost: K / R (per period)
        2. Holding cost: h × E[Inventory]
        3. Stockout cost: p × E[Stockouts]
        """
        protection_interval = review_period + self.L
        
        # Ordering cost per period
        ordering_cost = self.K / review_period
        
        # Expected inventory components
        cycle_stock = (self.mu * review_period) / 2  # Average cycle stock
        safety_stock = self.z_alpha * self.sigma * sqrt(protection_interval)  # Safety stock
        expected_inventory = cycle_stock + safety_stock
        
        # Holding cost
        holding_cost = self.h * expected_inventory
        
        # Stockout cost using loss function
        demand_std = self.sigma * sqrt(protection_interval)
        normalized_shortage = (base_stock_level - self.mu * protection_interval) / demand_std
        
        # Standard normal loss function: L(z) = φ(z) - z × (1 - Φ(z))
        loss_function = demand_std * (norm.pdf(normalized_shortage) - 
                                     normalized_shortage * (1 - norm.cdf(normalized_shortage)))
        
        stockout_cost = (self.p / review_period) * loss_function
        
        total_cost = ordering_cost + holding_cost + stockout_cost
        
        return total_cost, ordering_cost, holding_cost, stockout_cost
    
    def analytical_solution(self, review_period):
        """
        Compute analytical solution for a given review period.
        """
        # Calculate optimal base-stock level
        base_stock, pi, exp_demand, demand_std = self.calculate_base_stock_level(review_period)
        
        # Calculate cost components
        total_cost, ord_cost, hold_cost, stock_cost = self.calculate_total_cost(review_period, base_stock)
        
        # Calculate service level verification
        z_actual = (base_stock - exp_demand) / demand_std
        service_level = norm.cdf(z_actual)
        
        return {
            'review_period': review_period,
            'base_stock_level': base_stock,
            'protection_interval': pi,
            'expected_demand': exp_demand,
            'demand_std': demand_std,
            'total_cost': total_cost,
            'ordering_cost': ord_cost,
            'holding_cost': hold_cost,
            'stockout_cost': stock_cost,
            'service_level': service_level
        }

print("BaseStockOptimizer class defined successfully!")

BaseStockOptimizer class defined successfully!


In [ ]:
# Initialize the optimizer with MedSupply syringe data
optimizer = BaseStockOptimizer(
    demand_mean=12000,      # μ = 12,000 units/week
    demand_std=2400,        # σ = 2,400 units/week
    lead_time=0.5,          # L = 0.5 weeks
    holding_cost=0.15,      # h = $0.15/unit/week
    ordering_cost=75,       # K = $75/order
    stockout_cost=8,        # p = $8/unit
    service_level=0.95      # α = 95% service level
)

Initialized Base Stock Optimizer:
  Demand: N(12,000, 2,400²)
  Lead time: 0.5 weeks
  Service level: 0.950 (z = 1.645)
  Costs: h=$0.15, K=$75, p=$8


In [ ]:
# Calculate analytical solution for 1-week review period
solution_1_week = optimizer.analytical_solution(review_period=1.0)

print("=" * 60)
print("ANALYTICAL SOLUTION FOR 1-WEEK REVIEW PERIOD")
print("=" * 60)
print(f"Review Period: {solution_1_week['review_period']:.2f} weeks")
print(f"Protection Interval: {solution_1_week['protection_interval']:.2f} weeks")
print(f"Expected Demand: {solution_1_week['expected_demand']:,.0f} units")
print(f"Demand Std Dev: {solution_1_week['demand_std']:,.0f} units")
print()
print(f"Optimal Base-Stock Level: {solution_1_week['base_stock_level']:,.0f} units")
print(f"Achieved Service Level: {solution_1_week['service_level']:.4f} ({solution_1_week['service_level']*100:.2f}%)")
print()
print("COST BREAKDOWN:")
print(f"Ordering Cost: ${solution_1_week['ordering_cost']:,.2f}/week")
print(f"Holding Cost: ${solution_1_week['holding_cost']:,.2f}/week")
print(f"Stockout Cost: ${solution_1_week['stockout_cost']:,.2f}/week")
print(f"─" * 30)
print(f"Total Cost: ${solution_1_week['total_cost']:,.2f}/week")
print()

# Manual calculation verification
print("MANUAL CALCULATION VERIFICATION:")
print("Step 1: Base-Stock Level")
print(f"S* = μ(R+L) + zα × σ × √(R+L)")
print(f"S* = {12000} × (1+{0.5}) + {1.645} × {2400} × √(1+{0.5})")
print(f"S* = {12000*1.5:,.0f} + {1.645*2400*1.225:,.0f}")
print(f"S* = {solution_1_week['base_stock_level']:,.0f} units ✓")
print()
print("Step 2: Cost Components")
cycle_stock = 12000 * 1 / 2
safety_stock = 1.645 * 2400 * 1.225
print(f"Average Cycle Stock: {cycle_stock:,.0f} units")
print(f"Safety Stock: {safety_stock:,.0f} units")
print(f"Expected Inventory: {cycle_stock + safety_stock:,.0f} units")

ANALYTICAL SOLUTION FOR 1-WEEK REVIEW PERIOD
Review Period: 1.00 weeks
Protection Interval: 1.50 weeks
Expected Demand: 18,000 units
Demand Std Dev: 2,939 units

Optimal Base-Stock Level: 22,835 units
Achieved Service Level: 0.9500 (95.00%)

COST BREAKDOWN:
Ordering Cost: $75.00/week
Holding Cost: $1,625.23/week
Stockout Cost: $491.30/week
──────────────────────────────
Total Cost: $2,191.53/week

MANUAL CALCULATION VERIFICATION:
Step 1: Base-Stock Level
S* = μ(R+L) + zα × σ × √(R+L)
S* = 12000 × (1+0.5) + 1.645 × 2400 × √(1+0.5)
S* = 18,000 + 4,836
S* = 22,835 units ✓

Step 2: Cost Components
Average Cycle Stock: 6,000 units
Safety Stock: 4,836 units
Expected Inventory: 10,836 units


In [ ]:
# Compare different review periods
review_periods = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
comparison_results = []

for r in review_periods:
    result = optimizer.analytical_solution(r)
    comparison_results.append(result)

# Create comparison table
comparison_df = pd.DataFrame(comparison_results)
comparison_display = comparison_df[['review_period', 'base_stock_level', 'total_cost', 
                                   'ordering_cost', 'holding_cost', 'stockout_cost', 'service_level']]

print("COMPARISON OF DIFFERENT REVIEW PERIODS")
print("=" * 80)
print(comparison_display.round(2).to_string(index=False))

# Find optimal review period
optimal_idx = comparison_df['total_cost'].idxmin()
optimal_solution = comparison_df.iloc[optimal_idx]

print()
print("OPTIMAL SOLUTION:")
print(f"Review Period: {optimal_solution['review_period']:.2f} weeks")
print(f"Base-Stock Level: {optimal_solution['base_stock_level']:,.0f} units")
print(f"Total Cost: ${optimal_solution['total_cost']:,.2f}/week")
print(f"Service Level: {optimal_solution['service_level']:.4f}")

COMPARISON OF DIFFERENT REVIEW PERIODS
 review_period  base_stock_level  total_cost  ordering_cost  holding_cost  stockout_cost  service_level
           0.5          15947.65     1994.44          150.0       1042.15         802.29           0.95
           1.0          22834.86     2191.53           75.0       1625.23         491.30           0.95
           1.5          29582.82     2615.63           50.0       2187.42         378.20           0.95
           2.0          36241.78     3090.90           37.5       2736.27         317.13           0.95
           2.5          42837.53     3583.55           30.0       3275.63         277.92           0.95
           3.0          49385.37     4082.96           25.0       3807.81         250.16           0.95

OPTIMAL SOLUTION:
Review Period: 0.50 weeks
Base-Stock Level: 15,948 units
Total Cost: $1,994.44/week
Service Level: 0.9500


In [ ]:
# Create comprehensive visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Periodic Review Base-Stock Policy Analysis', fontsize=16, fontweight='bold')

# Plot 1: Cost Components vs Review Period
ax1.plot(comparison_df['review_period'], comparison_df['ordering_cost'], 
         'o-', label='Ordering Cost', linewidth=2, markersize=6)
ax1.plot(comparison_df['review_period'], comparison_df['holding_cost'], 
         's-', label='Holding Cost', linewidth=2, markersize=6)
ax1.plot(comparison_df['review_period'], comparison_df['stockout_cost'], 
         '^-', label='Stockout Cost', linewidth=2, markersize=6)
ax1.plot(comparison_df['review_period'], comparison_df['total_cost'], 
         'D-', label='Total Cost', linewidth=3, markersize=8, color='red')
ax1.set_xlabel('Review Period (weeks)')
ax1.set_ylabel('Cost ($/week)')
ax1.set_title('Cost Components vs Review Period')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Base-Stock Level vs Review Period
ax2.plot(comparison_df['review_period'], comparison_df['base_stock_level'], 
         'o-', linewidth=2, markersize=8, color='purple')
ax2.set_xlabel('Review Period (weeks)')
ax2.set_ylabel('Base-Stock Level (units)')
ax2.set_title('Optimal Base-Stock Level vs Review Period')
ax2.grid(True, alpha=0.3)

# Plot 3: Service Level vs Review Period
ax3.plot(comparison_df['review_period'], comparison_df['service_level'] * 100, 
         'o-', linewidth=2, markersize=8, color='green')
ax3.axhline(y=95, color='red', linestyle='--', alpha=0.7, label='Target Service Level')
ax3.set_xlabel('Review Period (weeks)')
ax3.set_ylabel('Service Level (%)')
ax3.set_title('Service Level Achievement vs Review Period')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Cost Breakdown Pie Chart for Optimal Solution
optimal_costs = [
    optimal_solution['ordering_cost'],
    optimal_solution['holding_cost'], 
    optimal_solution['stockout_cost']
]
colors = ['#ff9999', '#66b3ff', '#99ff99']
labels = [f'Ordering (${optimal_solution["ordering_cost"]:.0f})',
          f'Holding (${optimal_solution["holding_cost"]:.0f})',
          f'Stockout (${optimal_solution["stockout_cost"]:.0f})']

ax4.pie(optimal_costs, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax4.set_title(f'Cost Breakdown\n(R={optimal_solution["review_period"]:.1f} weeks)')

plt.tight_layout()
plt.show()

print("Visualization complete! The analysis shows:")
print(f"1. Total cost is minimized at review period of {optimal_solution['review_period']:.1f} weeks")
print(f"2. Base-stock level increases with review period to maintain service level")
print(f"3. Service level remains above target across all review periods")
print(f"4. Cost composition shifts as review period changes")


HEURISTIC OPTIMIZATION RESULTS
Optimal Review Period: 0.57 weeks
Optimal Base-Stock Level: 16,923 units
Minimum Total Cost: $1,985.08 per week

COST BREAKDOWN:
Ordering Cost: $131.58/week (6.6%)
Holding Cost: $1,125.52/week (56.7%)
Stockout Cost: $727.98/week (36.7%)

COMPARISON WITH TIER 1 (1-week review):
Tier 1 Cost: $2,328.06/week
Heuristic Cost: $1,985.08/week
Cost Savings: $342.98/week (14.7% reduction)
Review Period Change: 1.00 → 0.57 weeks


In [ ]:
# Sensitivity Analysis
print("SENSITIVITY ANALYSIS")
print("=" * 50)

# Test different service levels
service_levels = [0.90, 0.95, 0.98, 0.99]
sensitivity_results = []

for sl in service_levels:
    temp_optimizer = BaseStockOptimizer(
        demand_mean=12000, demand_std=2400, lead_time=0.5,
        holding_cost=0.15, ordering_cost=75, stockout_cost=8,
        service_level=sl
    )
    result = temp_optimizer.analytical_solution(1.0)
    sensitivity_results.append({
        'service_level': sl,
        'z_value': norm.ppf(sl),
        'base_stock': result['base_stock_level'],
        'total_cost': result['total_cost'],
        'safety_stock': result['base_stock_level'] - 12000 * 1.5
    })

sensitivity_df = pd.DataFrame(sensitivity_results)
print(sensitivity_df.round(2).to_string(index=False))

print()
print("KEY INSIGHTS:")
print("• Higher service levels require significantly more safety stock")
print("• The relationship between service level and inventory is non-linear")
print("• Moving from 95% to 99% service level requires substantial additional inventory")
print("• Cost increases with service level due to higher holding costs")

SENSITIVITY ANALYSIS
Initialized Base Stock Optimizer:
  Demand: N(12,000, 2,400²)
  Lead time: 0.5 weeks
  Service level: 0.900 (z = 1.282)
  Costs: h=$0.15, K=$75, p=$8
Initialized Base Stock Optimizer:
  Demand: N(12,000, 2,400²)
  Lead time: 0.5 weeks
  Service level: 0.950 (z = 1.645)
  Costs: h=$0.15, K=$75, p=$8

Initialized Base Stock Optimizer:
  Demand: N(12,000, 2,400²)
  Lead time: 0.5 weeks
  Service level: 0.980 (z = 2.054)
  Costs: h=$0.15, K=$75, p=$8
Initialized Base Stock Optimizer:
  Demand: N(12,000, 2,400²)
  Lead time: 0.5 weeks
  Service level: 0.990 (z = 2.326)
  Costs: h=$0.15, K=$75, p=$8
📈 Progress: 9/18 (50.0%)
 service_level  z_value  base_stock  total_cost  safety_stock
          0.90     1.28    21766.98     2653.33       3766.98
          0.95     1.64    22834.86     2191.53       4834.86
          0.98     2.05    24036.76     2053.19       6036.76
          0.99     2.33    24838.04     2080.39       6838.04

KEY INSIGHTS:
• Higher service levels requ

In [ ]:
# Summary and Conclusions
print("=" * 70)
print("PERIODIC REVIEW BASE-STOCK POLICY - TIER 1 SUMMARY")
print("=" * 70)
print()
print("PROBLEM OVERVIEW:")
print("• MedSupply Distribution manages syringe inventory with periodic review")
print("• Demand: N(12,000, 2,400²) units per week")
print("• Lead time: 0.5 weeks, Target service level: 95%")
print()
print("OPTIMAL POLICY PARAMETERS:")
print(f"• Review Period: {optimal_solution['review_period']:.2f} weeks")
print(f"• Base-Stock Level: {optimal_solution['base_stock_level']:,.0f} units")
print(f"• Total Expected Cost: ${optimal_solution['total_cost']:,.2f}/week")
print()
print("COST BREAKDOWN:")
print(f"• Ordering Cost: ${optimal_solution['ordering_cost']:,.2f}/week ({optimal_solution['ordering_cost']/optimal_solution['total_cost']*100:.1f}%)")
print(f"• Holding Cost: ${optimal_solution['holding_cost']:,.2f}/week ({optimal_solution['holding_cost']/optimal_solution['total_cost']*100:.1f}%)")
print(f"• Stockout Cost: ${optimal_solution['stockout_cost']:,.2f}/week ({optimal_solution['stockout_cost']/optimal_solution['total_cost']*100:.1f}%)")
print()
print("PERFORMANCE METRICS:")
print(f"• Service Level: {optimal_solution['service_level']*100:.2f}% (target: 95%)")
print(f"• Protection Interval: {optimal_solution['protection_interval']:.2f} weeks")
print(f"• Expected Demand: {optimal_solution['expected_demand']:,.0f} units")
print()
print("KEY MATHEMATICAL INSIGHTS:")
print("• Base-stock level balances expected demand and safety stock")
print("• Safety stock depends on demand variability and service level")
print("• Trade-off between ordering frequency and inventory holding")
print("• Service level constraint drives safety stock requirements")
print()
print("PRACTICAL IMPLICATIONS:")
print("• Weekly review provides good balance of cost and service")
print("• Safety stock represents ~21% of total base-stock level")
print("• Holding costs dominate total cost structure")
print("• Policy achieves target service level with minimal cost")

PERIODIC REVIEW BASE-STOCK POLICY - TIER 1 SUMMARY

PROBLEM OVERVIEW:
• MedSupply Distribution manages syringe inventory with periodic review
• Demand: N(12,000, 2,400²) units per week
• Lead time: 0.5 weeks, Target service level: 95%

OPTIMAL POLICY PARAMETERS:
• Review Period: 0.50 weeks
• Base-Stock Level: 15,948 units
• Total Expected Cost: $1,994.44/week

COST BREAKDOWN:
• Ordering Cost: $150.00/week (7.5%)
• Holding Cost: $1,042.15/week (52.3%)
• Stockout Cost: $802.29/week (40.2%)

PERFORMANCE METRICS:
• Service Level: 95.00% (target: 95%)
• Protection Interval: 1.00 weeks
• Expected Demand: 12,000 units

KEY MATHEMATICAL INSIGHTS:
• Base-stock level balances expected demand and safety stock
• Safety stock depends on demand variability and service level
• Trade-off between ordering frequency and inventory holding
• Service level constraint drives safety stock requirements

PRACTICAL IMPLICATIONS:
• Weekly review provides good balance of cost and service
• Safety stock represents